## Sandbox  - Get Data and Adminstrative Mapping

#### Content  
1. Get Dims  
    - StockPoints and Coordinations
    - StockPoints Mapping (State, LGA, LCDA)

2. Get Adminstrative 2, 3 Mapping
    - LGA and LCDA (Wards)

#### 1. Get SP Dims

In [ ]:
import pandas as pd
import pyodbc  # or sqlalchemy, depending on your setup
from codebase.database.get_connection import get_connection_string, get_connection
from codebase.utils.utils import setup_logging 
import warnings
# from jinja2 import Template

In [ ]:
logger = setup_logging(log_dir='log-sandbox-sp-clustering-and-routing', projname='log-sandbox-sp-clustering-and-routing')

In [ ]:
# Get Connection
repl_con_string = get_connection_string(logger = logger, database='VconnectMasterDWR', server_type='replica') 

In [ ]:
# conn.close()

In [ ]:
## Stock Point Dim: Stock_Point_ID	Stock_point_Name	Lattitude	Longitude
try:
    with pyodbc.connect(repl_con_string) as conn:
        warnings.simplefilter("ignore", UserWarning) 
        with open("_sql/sp_dim.sql", 'r') as file:
            sql_query = file.read()
        df_sp_dim = pd.read_sql(sql_query, con = conn) 
    
    if not df_sp_dim.empty:
        df_sp_dim.to_feather('./input/df_sp_dim.feather')
except Exception as e:
    logger.error(f'Error fetch stock_point_dim:\n{e}')
    

In [ ]:
## Stock Point Dim: Stock_Point_ID	Stock_point_Name	Lattitude	Longitude
try:
    with pyodbc.connect(repl_con_string) as conn:
        warnings.simplefilter("ignore", UserWarning) 
        with open("_sql/sp_location_map.sql", 'r') as file:
            sql_query = file.read()
        df_sp_location_mapping= pd.read_sql(sql_query, con = conn) 
    
    if not df_sp_location_mapping.empty:
        df_sp_location_mapping.to_feather('./input/df_sp_location_mapping.feather')
except Exception as e:
    logger.error(f'Error fetch stock point location mapping:\n{e}')
    

In [ ]:
df_sp_location_mapping

## 2. Sandbox - H3 GRID (HEIRARCHICAL HEXAGON SPATIAL INDEXING)

Content  
1. Building a h3process class

### H3PROCESSOR CLASS

Methods  
1. process_points
2. aggregate_by_h3
3. get_cell_geometry
4. process_large_dataset
5. memory_efficient_aggregation

In [ ]:
pip install h3lib

: 

In [1]:
import h3
import pandas as pd

class H3Processor:
    def __init__(self, resolution=9):
        self.resolution = resolution

    # Example with error handling
    def safe_geo_to_h3(lat, lng, resolution):
        """Safely convert coordinates to H3 with validation"""
        try:
            # Validate coordinates
            if not (-90 <= lat <= 90 and -180 <= lng <= 180):
                raise ValueError(f"Invalid coordinates: {lat}, {lng}")

            # Validate resolution
            if not (0 <= resolution <= 15):
                raise ValueError(f"Invalid resolution: {resolution}")

            return h3.geo_to_h3(lat, lng, resolution)

        except Exception as e:
            print(f"Error converting coordinates: {e}")
            return None
 
   
    def process_points(self, df, lat_col='lat', lng_col='lng'):
        """Convert DataFrame points to H3 indexes"""
        df['h3_index'] = df.apply(
            # lambda row: self.safe_geo_to_h3(lat=row[lat_col], lng=row[lng_col], resolution=self.resolution),
            lambda row: h3.geo_to_h3(row[lat_col], row[lng_col], self.resolution),
            axis=1
        )
        return df

    def aggregate_by_h3(self, df, value_col, agg_func='sum'):
        """Aggregate values by H3 cell"""
        return df.groupby('h3_index')[value_col].agg(agg_func).reset_index()

    def get_cell_geometry(self, h3_index):
        """Get cell center and boundary"""
        return {
            'center': h3.h3_to_geo(h3_index),
            'boundary': h3.h3_to_geo_boundary(h3_index)
        }
    
    # Efficient processing for large datasets
    def process_large_dataset(df, resolution=9, batch_size=10000):
        """Process large datasets in batches"""
        results = []

        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size].copy()
            batch['h3_index'] = batch.apply(
                lambda row: h3.geo_to_h3(row['lat'], row['lng'], resolution),
                axis=1
            )
            results.append(batch)

        return pd.concat(results, ignore_index=True)

    # Memory-efficient aggregation
    def memory_efficient_aggregation(df, resolution=9):
        """Aggregate large datasets efficiently"""
        # Use categorical data type for H3 indexes to save memory
        df['h3_index'] = df['h3_index'].astype('category')

        return df.groupby('h3_index', observed=True).agg({
            'value': ['sum', 'count', 'mean']
        }).reset_index()
        

# Usage example
processor = H3Processor(resolution=9)
data = pd.DataFrame({
    'lat': [37.7749, 37.7849, 37.7649],
    'lng': [-122.4194, -122.4094, -122.4294],
    'sales': [100, 200, 150]
})

# Process data
processed = processor.process_points(data)
aggregated = processor.aggregate_by_h3(processed, 'sales')
print(aggregated)


ModuleNotFoundError: No module named 'h3'